# Grover search

Amplify the marked two-qubit state and compare the complete ideal distribution.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    qiskit_selection,
    total_variation_distance,
)

In [ ]:
circuit = QuantumCircuit(2)
circuit.h(range(2))
circuit.cz(0, 1)  # mark |11>
circuit.h(range(2))
circuit.x(range(2))
circuit.cz(0, 1)
circuit.x(range(2))
circuit.h(range(2))

def get_reference():
    return Statevector.from_instruction(circuit).probabilities()

reference, reference_ms, _ = benchmark(get_reference)
backend = MettleQBackend(method="statevector", device="cpu")
compiled = transpile(circuit, backend, optimization_level=1)

def get_mettleq():
    state = backend.run(compiled, shots=1, return_statevector=True).result().data(0)["statevector"]
    return np.abs(np.asarray(state)) ** 2

candidate, mettleq_ms, _ = benchmark(get_mettleq)
error = max_abs_error(reference, candidate)
marked = format(int(np.argmax(candidate)), "02b")
method, device = qiskit_selection(backend)
tutorial_result = emit_result(
    notebook="qiskit/07_grover_search.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="probability vector atol=2e-6 and exact marked item",
    passed=error <= 2e-6 and marked == "11",
    exact_match=marked == "11",
    selected_method=method,
    selected_device=device,
    metrics={"max_probability_error": error, "marked_item": marked, "marked_probability": candidate[-1]},
)